# Problem Set 8: Project Genesis – Der souveräne Wächter

Sensible Telemetriedaten, physikalische Anomalien und geschützte Systemzustände wurden über unsichere Netzwerke an externe Server übertragen. Jede Millisekunde Latenz bedrohte die Stabilität eurer Echtzeitschleifen; jeder Netzwerkausfall führte zum sofortigen Zusammenbruch eures digitalen Abbilds.

Heute errichten wir die **Festung der Autarkie**. Ihr werdet ein hochentwickeltes, lokal lauffähiges Modell der **Google Gemma 4**-Familie tief im Fundament eurer eigenen Hardware verankern. Eure Mission ist es, eine ereignisdiskrete Warteschlangensimulation – das pulsierende Herz einer virtuellen Fabrikationsstraße – mit diesem lokalen Wächter zu koppeln. Ohne ein einziges Datenpaket nach außen zu senden, wird der Wächter die Telemetrie überwachen, im geschützten Denkanal logische Analysen durchführen, auf eine lokale Wissensdatenbank zugreifen und regulierend in die Physik der Warteschlange eingreifen.

Bereitet eure Terminals vor. Aktiviert die lokalen Tensor-Kerne. Die Souveränität über die Daten ist der Schlüssel zur absoluten Kontrolle. Let's build!

## Exercise 1: Die Initialisierung des lokalen Wächters (Gemma 4 Edge)

Wir verwenden die hochgradig effiziente Variante **Gemma 4 E2B-IT**. Dieses Modell weist trotz seiner kompakten Größe (2,3 Milliarden effektive Parameter) dank der neuartigen Per-Layer Embeddings (PLE) eine exzellente Logikfähigkeit auf.

### Instruktionen:
1. Melden Sie sich bei [Kaggle](https://www.kaggle.com) an und akzeptieren Sie die Nutzungsbedingungen auf der offiziellen Gemma 4 Modellseite.
2. Erstellen Sie ein API-Token in Ihren Kaggle-Kontoeinstellungen, um die Datei `kaggle.json` herunterzuladen.
3. Setzen Sie die Umgebungsvariablen `KAGGLE_USERNAME` und `KAGGLE_KEY` in Ihrer Python-Laufzeitumgebung.
4. Laden Sie das Modell mit `kagglehub` herunter und initialisieren Sie die Inferenz-Pipeline.

In [ ]:
import os
import kagglehub

# Tragen Sie hier Ihre Credentials ein (oder nutzen Sie eine .env Datei)
# os.environ["KAGGLE_USERNAME"] = "ihr_benutzername"
# os.environ["KAGGLE_KEY"] = "ihr_api_schluessel"

try:
    print("Fordere Gewichte von der Kaggle-Schnittstelle an...")
    # Herunterladen der instruction-tuned Gemma 4 E2B Variante über kagglehub
    model_path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")
    print(f"Modell erfolgreich lokalisiert im Systemcache unter: {model_path}")
except Exception as e:
    print(f"Fehler beim Herunterladen des Modells: {e}")
    print("Sicherstellen, dass die Modell-Lizenz auf Kaggle akzeptiert wurde und die API-Schlüssel korrekt konfiguriert sind.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Initialisiere Inferenz-Pipeline (bfloat16 für VRAM-Effizienz)...")
try:
    processor = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    print("Lokaler Wächter online und inferenzbereit.")
except NameError:
    print("Modellpfad nicht gefunden. Führen Sie die obige Zelle zuerst aus.")

**Frage:** Warum ist diese Einsparung (Kompakte Parameter/bfloat16) ein entscheidender Faktor bei der Ausführung paralleler physikalischer Simulationen auf derselben Hardware?

*Ihre Antwort hier eintragen:*

## Exercise 2: Aktivierung des Cognitive Core (Der Denkmodus)

Dieser Prozess wird nativ gestartet, indem das Token `<|think|>` am Anfang der Systemnachricht platziert wird. Das Modell gibt seine logischen Gedankenschritte innerhalb der Trenner `<|channel>thought\n` und `<channel|>` aus.

### Instruktionen:
1. Implementieren Sie eine Inferenz-Funktion `generate_thoughtful_response`, die den System-Prompt so formatiert, dass der native Denkmodus aktiviert wird.
2. Nutzen Sie reguläre Ausdrücke (RegEx), um den Denkanal (`thought`) von der finalen Antwort zu trennen.
3. Schreiben Sie eine Hilfsfunktion, die die Gedankenblöcke aus der Historie entfernt, bevor die nächste Benutzerinteraktion stattfindet.

In [ ]:
import re

def generate_thoughtful_response(system_prompt, user_message, max_new_tokens=512):
    messages = [
        {"role": "system", "content": f"<|think|>\n{system_prompt}"},
        {"role": "user", "content": user_message}
    ]
    
    # Erstellung des Chat-Templates über den integrierten Prozessor
    prompt_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Konvertierung in Eingabe-Tensors
    inputs = processor(text=prompt_text, return_tensors="pt").to(model.device)
    
    # Inferenz ohne Gradientenberechnung zur Performance-Optimierung
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministische Ausgabe für reproduzierbare Systemreaktionen
            eos_token_id=processor.eos_token_id
        )
    
    # Isolation der generierten Tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    decoded_output = processor.decode(generated_tokens, skip_special_tokens=False)
    
    # RegEx-Muster zur Isolation des Denkanals
    thought_pattern = r"<\|channel>thought\s*(.*?)\s*<channel|>"
    thought_match = re.search(thought_pattern, decoded_output, re.DOTALL)
    
    thought_content = thought_match.group(1).strip() if thought_match else "Kein expliziter Denkprozess aufgezeichnet."
    
    # Bereinigung der Antwort für den finalen Befehlscode
    clean_response = re.sub(thought_pattern, "", decoded_output, flags=re.DOTALL).strip()
    clean_response = clean_response.replace("<turn|>", "").replace("<bos>", "").replace("<eos>", "").strip()
    
    return thought_content, clean_response

In [ ]:
def clean_history(messages):
    """
    Hilfsfunktion: Entfernt Gedankenblöcke aus der Historie (State-Management).
    Dies verhindert eine kognitive Überladung und Token-Wachstum bei Multi-Turn Chats.
    """
    cleaned_messages = []
    thought_pattern = r"<\|channel>thought\s*.*?\s*<channel|>"
    for msg in messages:
        if msg["role"] == "model":
            clean_content = re.sub(thought_pattern, "", msg["content"], flags=re.DOTALL).strip()
            cleaned_messages.append({"role": "model", "content": clean_content})
        else:
            cleaned_messages.append(msg)
    return cleaned_messages

## Exercise 3: Die diskrete Warteschlangensimulation (Foundry)

Teile kommen an, warten im Puffer und werden von einer Maschine abgearbeitet. Wir simulieren dieses Verhalten stochastisch. Die Ankunftszeiten folgen einem Poisson-Prozess mit Rate $\lambda$ (Arrival Rate), die Bearbeitungszeiten an der Station folgen einer Exponential- bzw. Poisson-Verteilung mit Rate $\mu$ (Service Rate).

### Instruktionen:
1. Schreiben Sie eine Simulationsklasse `DiscreteFoundry`, die den Zustand eines Puffers abbildet.
2. Wenn die Ankunftsrate die Bearbeitungsrate übersteigt (Instabilität), füllt sich der Puffer.
3. Bei Erreichen der maximalen Kapazität kommt es zum Pufferüberlauf (Datenverlust/Ausschuss).
4. Die Methode `step()` muss die Systemvariablen aktualisieren und einen JSON-String exportieren.

In [ ]:
import numpy as np
import json

class DiscreteFoundry:
    def __init__(self, arrival_rate=2.5, service_rate=1.0, capacity=15):
        self.arrival_rate = arrival_rate
        self.service_rate = service_rate
        self.capacity = capacity
        self.buffer = 0
        self.dropped = 0
        self.processed = 0
        
    def step(self):
        # Stochastische Approximation
        arrivals = np.random.poisson(self.arrival_rate)
        services = np.random.poisson(self.service_rate)
        
        self.buffer += arrivals
        
        # Überlauf prüfen
        if self.buffer > self.capacity:
            overflow = self.buffer - self.capacity
            self.dropped += overflow
            self.buffer = self.capacity
            
        # Verarbeitung
        processed_now = min(self.buffer, services)
        self.buffer -= processed_now
        self.processed += processed_now
        
        return {
            "queue_length": self.buffer,
            "max_capacity": self.capacity,
            "dropped_parts": self.dropped,
            "processed_parts": self.processed,
            "current_service_rate": self.service_rate
        }

In [ ]:
# Instanziierung der Produktionslinie
production_line = DiscreteFoundry()
print("Initialer Tick:", json.dumps(production_line.step()))

## Exercise 4: Geschlossener Regelkreis mit kognitiver Instanz

Die Telemetriedaten werden an den lokalen Wächter übergeben. Dieser analysiert die Situation offline im Denkanal, trifft eine kognitive Entscheidung und gibt einen neuen Wert für die Service-Rate vor, welcher unmittelbar in die Simulation zurückgeführt wird.

### Instruktionen:
1. Konzipieren Sie einen System-Prompt für Gemma 4, der das Modell anweist, als industrieller Regler zu agieren.
2. Das Modell muss den Systemzustand bewerten und die Service-Rate anpassen (Intervall: $[0.5, 3.0]$).
3. Enforcieren Sie ein klares Ausgabe-Format, um den neuen Parameter numerisch parsen zu können.
4. Lassen Sie die gesteuerte Simulation über 8 Zeitschritte laufen.

In [ ]:
sentinel_system_prompt = """
Du bist der souveräne Wächter (KI-Regler) einer industriellen Fabrikationsstraße.
Dein Ziel ist es, Pufferüberläufe (dropped_parts) zu minimieren, aber gleichzeitig Energie zu sparen.
Maximales Intervall für Service-Rate: 0.5 bis 3.0.
Analysiere die Telemetrie: Wenn queue_length hoch ist, erhöhe die Rate drastisch. Wenn niedrig, senke sie.
Gib AUSSCHLIESSLICH den neuen Wert in der letzten Zeile im exakten Format aus: 
NEW_SERVICE_RATE: <wert>
"""

print("=== START DES GESCHLOSSENEN REGELKREISES ===")
try:
    for tick in range(8):
        # 1. Systemschritt
        telemetry_data = production_line.step()
        print(f"\n[Tick {tick+1}] Telemetrie-Eingang: {json.dumps(telemetry_data)}")
        
        # 2. Prompt-Generierung
        user_query = f"Statusbericht: {json.dumps(telemetry_data)}\nAktuell eingestellte Service-Rate: {production_line.service_rate}. Entscheidung?"
        
        # 3. Lokale Inferenz
        thought_process, action_response = generate_thoughtful_response(sentinel_system_prompt, user_query, max_new_tokens=256)
        
        print(f"💭 [Denkanal]: {thought_process[:200]}... [gekürzt]")
        print(f"🤖 [Modell-Entscheidung]: {action_response}")
        
        # 4. Parsing der Entscheidung
        match = re.search(r"NEW_SERVICE_RATE:\s*([0-9.]+)", action_response)
        if match:
            proposed_rate = float(match.group(1))
            # Physikalische Absicherung über System-Schranken
            validated_rate = max(0.5, min(3.0, proposed_rate))
            production_line.service_rate = validated_rate
            print(f">>> System-Aktion: Service-Rate für Schritt {tick+2} auf {validated_rate} aktualisiert.")
        else:
            print("!!! Parsing-Fehler: Keine gültige Anpassung gefunden. Behalte aktuellen Parameter bei.")
except NameError:
    print("Das Modell wurde in Zelle 4 noch nicht erfolgreich in den Speicher geladen.")

## Exercise 5: Cloud-gestütztes Fine-Tuning über die Colab CLI

Um das daten- und rechenintensive Modell-Training auszulagern, ohne den lokalen Terminal-Workflow zu unterbrechen, nutzen wir die neue **Google Colab CLI** (`google-colab-cli`). Sie ermöglicht es uns, über ein einziges Kommando Remote-GPUs im Google-Rechenzentrum zu provisionieren, Skripte auszuführen und Trainings-Ergebnisse (LoRA-Adapter) direkt in unseren lokalen Workspace herunterzuladen.

### Szenario:
Wir wollen ein feingetuntes **Gemma 4 E2B**-Modell über QLoRA trainieren, damit es Telemetrieberichte direkt in formstabilen industriellen Steuerungscodes formatiert.

### Instruktionen:
1. Installieren Sie die offizielle Google Colab CLI in Ihrer Terminal-Umgebung.
2. Erstellen Sie ein lokales Python-Skript `finetune_run.py`, das Gemma 4 unter Verwendung von KerasNLP/Transformers für ein LoRA-Feintuning initialisiert.
3. Verwenden Sie die Colab CLI-Befehlssequenz, um ein Remote-System mit einer T4 GPU anzufordern, die Pakete zu installieren, das Skript auszuführen und die fertig trainierten Gewichts-Adapter (`safetensors`) zurück in Ihren lokalen Ordner zu übertragen.

In [ ]:
# Optionale Installation der CLI im lokalen Terminal:
# pip install google-colab-cli

In [ ]:
finetune_script = """
import keras
import keras_hub
import os

os.environ["KERAS_BACKEND"] = "jax"

print("Lade Gemma 4 E2B für LoRA Fine-Tuning...")
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")

# 2. Aktivierung von Low-Rank Adaptation (LoRA)
gemma_lm.backbone.enable_lora(rank=4)

# 3. Dummy-Trainingsdaten formatieren
training_data = [
    {"prompt": "Status: queue_length=14. Action?", "response": "NEW_SERVICE_RATE: 3.0"},
    {"prompt": "Status: queue_length=1. Action?", "response": "NEW_SERVICE_RATE: 0.5"}
]
prompts = [f"{item['prompt']} -> {item['response']}" for item in training_data]

# 4. Kompilierung des Modells mit optimierter Lernrate
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    weighted_metrics=["accuracy"]
)

# 5. Training für eine repräsentative Epoche
gemma_lm.fit(prompts, epochs=1, batch_size=2)

# 6. Sichern des trainierten Adapters im VM-Speicher
gemma_lm.save_to_preset("./gemma4_lora_adapter")
print("Feintuning erfolgreich abgeschlossen. Gewichte lokal in der Colab-VM gespeichert.")
"""

with open("finetune_run.py", "w") as f:
    f.write(finetune_script.strip())
print("Lokales Skript 'finetune_run.py' erstellt und bereit zum Versand!")

Das folgende Kommando veranschaulicht den transparenten Code-Uplink in der Bash-Shell:

```bash
# 1. Allokieren Sie eine neue Remote-VM mit einer performanten T4-GPU
colab new -s gemma-tuning --gpu T4

# 2. Installieren Sie die modernsten ML-Bibliotheken direkt auf der Remote-Instanz via 'uv'
colab install -s gemma-tuning keras-hub keras tensorflow torch

# 3. Führen Sie Ihr lokales Python-Skript direkt auf der remote GPU aus (Transparenter Code-Uplink)
colab exec -s gemma-tuning -f finetune_run.py

# 4. Laden Sie das Ergebnis (den fertigen LoRA-Gewichts-Adapter) sicher herunter
colab download -s gemma-tuning /content/gemma4_lora_adapter ./local_gemma4_adapter

# 5. Beenden Sie die Remote-Sitzung, um Ihre Rechenressourcen zu schonen
colab stop -s gemma-tuning
```

**Frage:** Wie schützt das Zusammenspiel aus Colab CLI und lokalem Edge-Modell die geistige Souveränität Ihrer Simulationsdaten?

*Ihre Antwort hier eintragen:*

## Exercise 6: Kontext-Injektion (Lokales RAG für SOPs)

Der Wächter muss spezifische Maschinenvorgaben (Standard Operating Procedures, SOPs) heranziehen, um den optimalen Modus auszuwählen. In dieser Übung implementieren Sie ein offline-fähiges RAG-System, das auf Basis der aktuellen Queue-Länge die passende Regelungsvorschrift aus einem lokalen Dokumentenindex extrahiert und als Kontext in das Gemma 4-Modell einspeist.

### Instruktionen:
1. Erstellen Sie eine Liste von SOPs als lokalen Text-Index.
2. Implementieren Sie eine einfache, abstandsbasierte Suchfunktion (z. B. Keyword-Matching oder Cosine-Similarity über Wortfrequenzen), um das relevanteste SOP-Dokument basierend auf der Queue-Länge zu filtern.
3. Integrieren Sie dieses Dokument als zusätzlichen Kontext in den Prompt für Gemma 4.

In [ ]:
# Lokaler RAG-Index
SOP_DATABASE = {
    "SOP-101_NORMAL": "Warteschlange < 5. Maschine im Eco-Modus belassen. Rate auf 1.0 drosseln.",
    "SOP-202_WARNING": "Warteschlange >= 5 und <= 10. Maschine in Normalbetrieb. Rate auf 2.0 erhöhen.",
    "SOP-999_CRITICAL": "Warteschlange > 10. NOTFALL-PROTOKOLL. Rate zwingend auf 3.0 setzen!"
}

def retrieve_relevant_sop(queue_length):
    if queue_length > 10:
        return SOP_DATABASE["SOP-999_CRITICAL"]
    elif queue_length >= 5:
        return SOP_DATABASE["SOP-202_WARNING"]
    else:
        return SOP_DATABASE["SOP-101_NORMAL"]

# Testlauf des lokalen RAG-Systems
test_queue = 12
retrieved_sop = retrieve_relevant_sop(test_queue)
print(f"Extrahierte SOP für Warteschlangenlänge {test_queue}:\n-> {retrieved_sop}")

In [ ]:
# Demonstration der RAG-gestützten Prompt-Injektion
augmented_prompt = f"{sentinel_system_prompt}\n\nWICHTIGE VORSCHRIFT (SOP) FÜR DIESEN SCHRITT:\n{retrieved_sop}"
demo_query = f"Statusbericht: Queue={test_queue}, Capacity=15. Aktuelle Service-Rate: 1.0. Entscheidung?"

try:
    t, a = generate_thoughtful_response(augmented_prompt, demo_query, max_new_tokens=256)
    print("\n=== RAG-AUGMENTIERTE ENTSCHEIDUNG ===")
    print("Gedanken:\n", t)
    print("\nAktion:\n", a)
except NameError:
    print("Bitte stellen Sie sicher, dass das Gemma-Modell in Zelle 4 geladen wurde.")

**Frage zur Reflexion:**
Inwiefern erleichtert dieser Ansatz die Anpassung der Systemlogik, wenn sich die Werksvorschriften ändern?

*Ihre Antwort hier eintragen:*